Metagenomics

1. Quality Control

In [ ]:
#(1) kneadata
source activate kneaddata
thread=3
dir_i=/your/input/raw/sequencing/file/folder

/data/home/quj_lab/xiaoyu/software/miniconda2/bin/parallel -j ${thread} \
kneaddata --input ${dir_i}/{1}/*/*_1.fq.gz --input ${dir_i}/{1}/*/*_2.fq.gz \
--threads 6 --processes 6 \
--reference-db /data/home/quj_lab/luxiaoyong/software/kneaddata_database/hg19_kneaddata_bowtie2_database \
--output ./{1} \
--trimmomatic /data/home/quj_lab/luxiaoyong/software/miniconda3/envs/kneaddata/share/trimmomatic-0.39-2 \
--remove-intermediate-output \
--run-fastqc-start \
--run-fastqc-end \
--output-prefix {1}\
--cat-final-output :::: /your/input/sample/list

2. Read-based metagenome workflow

In [ ]:
#(1) humann3
source activate humann3
thread=3

dir_i=/your/kneaddata/fastq/file/folder
dir_o=/your/output/file/folder

/data/home/quj_lab/xiaoyu/software/miniconda2/bin/parallel -j ${thread} \
/data/home/quj_lab/luxiaoyong/software/miniconda3/envs/humann3/bin/humann3 --threads 8\
            -i ${dir_i}/{1}/{1}.fastq \
            -o ${dir_o}/{1}  \
            --bowtie2 /data/home/quj_lab/luxiaoyong/software/miniconda3/envs/humann3/bin/bowtie2 \
            --diamond /data/home/quj_lab/luxiaoyong/software/miniconda3/envs/humann3/bin/diamond \
            --metaphlan-options="-t=rel_ab_w_read_stats" :::: /your/input/sample/list

3. Assembly-based metagenome workflow

In [ ]:
#(1) bbrename
source activate metawrap-env
thread=3
dir_i=/your/kneaddata/fastq/file/folder

/data/home/quj_lab/xiaoyu/software/miniconda2/bin/parallel -j ${thread} \
bbrename.sh in1=${dir_i}/{1}/{1}--cat-final-output_paired_1.fastq in2=${dir_i}/{1}/{1}--cat-final-output_paired_2.fastq \
out1=${dir_i}/{1}/{1}--cat-final-output_rename_paired_1.fastq \
out2=${dir_i}/{1}/{1}--cat-final-output_rename_paired_2.fastq :::: /your/input/sample/list
                
#(2) SPAdes
source activate metag
thread=3
dir_i=/your/kneaddata/fastq/file/folder

/data/home/quj_lab/xiaoyu/software/miniconda2/bin/parallel -j ${thread} \
spades.py --pe1-1 ${dir_i}/{1}/{1}--cat-final-output_paired_1.fastq --pe1-2 ${dir_i}/{1}/{1}--cat-final-output_paired_2.fastq \
--meta -m 250 -t 6 -o {1}  :::: /your/input/sample/list
                  
#(3) metawrap binning
source activate metag
dir_i=/your/SPAdes/fastq/file/folder
dir_fastq=/your/bbrename/fastq/file/folder

export text_path=/your/input/sample/list
for i in $(cat $text_path)
do
    metawrap binning -o ./${i}_INITIAL_BINNING -t 30 \
    -a ${dir_i}/${i}/contigs.fasta --metabat2 --maxbin2 --concoct ${dir_fastq}/${i}--cat-final-output_rename_paired_*.fastq 
done
            
#(4) metawrap bin refinement
source activate metawrap-env
dir_i=/your/metawrap/file/folder
cd ${dir_i}
samples=`ls . | sort -n`

for i in ${samples};do mkdir ${i};cd ${i};pwd;
    metawrap bin_refinement -o BIN_REFINEMENT -t 30 \
    -A ${dir_i}/${i}/metabat2_bins/ \
    -B ${dir_i}/${i}/maxbin2_bins/ \
    -C ${dir_i}/${i}/concoct_bins/ \
    -c 50 -x 10;done
      
#(5) checkm
source activate metawrap-env 
dir_i=/your/metawrap/bin/refinement/file/folder
cd ${dir_i}
samples=`ls . | sort -n`

for i in ${samples};do cd ${i}/metawrap_50_10_bins;checkm lineage_wf -t 8 -x fa ./ ./checkmout/  --tab_table -f checkm_result.tab;
cd ${dir_i};done
  
#(6) dRep_dereplicate
source activate metag
dir_i=/your/metawrap/bin/refinement/file/folder
cd ${dir_i}

dRep dereplicate ../dRep_dereplicate_output/ -g ./*.fa

#(7) GTDB
source activate gtdb-tk
dir_i=/your/metawrap/bin/refinement/file/folder
cd ${dir_i}
samples=`ls . | sort -n`

for i in ${samples};do cd ${i}/BIN_REFINEMENT/;gtdbtk classify_wf --genome_dir ./metawrap_50_10_bins --out_dir ./gtdbtk_classify_wf -x fa;
cd ${dir_i};done

Virulence factors

In [ ]:
source activate python2.7

dir_i=/dellstorage06/quj_lab/luxiaoyong/02_Result/QZ/01_metagenomics/02_Male/01_kneaddata
cd ${dir_i}
samples=`ls . | sort -n`

cd /dellstorage06/quj_lab/luxiaoyong/02_Result/QZ/01_metagenomics/02_Male/06_CARD

for i in ${samples}
do
shortbred_quantify.py --markers /dellstorage09/quj_lab/luxiaoyong/software/ShortBRED_database/ShortBRED_CARD_2017_markers.faa \
--wgs ${dir_i}/${i}/${i}.fastq \
--results ${i}_CARD.txt --tmp ./${i}_example_quantify --usearch /data/home/quj_lab/luxiaoyong/software/usearch
done

Antibiotic resistance genes

In [ ]:
source activate python2.7

dir_i=/dellstorage06/quj_lab/luxiaoyong/02_Result/QZ/01_metagenomics/02_Male/01_kneaddata
cd ${dir_i}
samples=`ls . | sort -n`
echo ${samples}

cd /dellstorage06/quj_lab/luxiaoyong/02_Result/QZ/01_metagenomics/02_Male/07_VF

for i in ${samples}
do
shortbred_quantify.py --markers /dellstorage09/quj_lab/luxiaoyong/software/ShortBRED_database/ShortBRED_VF_2017_markers.faa \
--wgs ${dir_i}/${i}/${i}.fastq \
--results ${i}_VF.txt --tmp ./${i}_example_quantify --usearch /data/home/quj_lab/luxiaoyong/software/usearch
done

Clinical data

Clinical data were exported directly. Categorical option data were coded as 1, 2, 3, 4 for analysis, and raw numerical data were unprocessed.